In [40]:
# read final_mock_dataset.csv into df 
import pandas as pd
df = pd.read_csv('../../data/processed/final_mock_dataset.csv')

# keep only rows where source is 'smart_campus_room_measurements'
df = df[df['source'] == 'smart_campus_room_measurements']
df

,timestamp,session_id,location_id,record_id,source,humidity,light,temperature,noise,co2,focus_score
775247,2025-09-15 00:00:03+00:00,smart_campus_room_measurements__room_2_12__s00001,room_2_12,smart_campus_room_measurements:row_00000001,smart_campus_room_measurements,59.0,NaN,24.4,38.0,413.0,NaN
775249,2025-09-15 00:05:05+00:00,smart_campus_room_measurements__room_2_12__s00001,room_2_12,smart_campus_room_measurements:row_00000002,smart_campus_room_measurements,59.0,NaN,24.5,39.0,418.0,NaN
775251,2025-09-15 00:10:08+00:00,smart_campus_room_measurements__room_2_12__s00001,room_2_12,smart_campus_room_measurements:row_00000003,smart_campus_room_measurements,59.0,NaN,24.4,39.0,421.0,NaN
775253,2025-09-15 00:15:11+00:00,smart_campus_room_measurements__room_2_12__s00001,room_2_12,smart_campus_room_measurements:row_00000004,smart_campus_room_measurements,59.0,NaN,24.4,38.0,413.0,NaN
775255,2025-09-15 00:20:14+00:00,smart_campus_room_measurements__room_2_12__s00001,room_2_12,smart_campus_room_measurements:row_00000005,smart_campus_room_measurements,59.0,NaN,24.4,38.0,413.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
910477,2025-12-19 23:54:06+00:00,smart_campus_room_measurements__room_2_12__s00001,room_2_12,smart_campus_room_measurements:row_00108937,smart_campus_room_measurements,44.0,NaN,21.9,58.0,509.0,NaN
910479,2025-12-19 23:55:37+00:00,smart_campus_room_measurements__lab_2_2__s00001,lab_2_2,smart_campus_room_measurements:row_00108938,smart_campus_room_measurements,52.0,NaN,18.4,49.0,542.0,NaN
910480,2025-12-19 23:58:37+00:00,smart_campus_room_measurements__room_2_5__s00001,room_2_5,smart_campus_room_measurements:row_00108939,smart_campus_room_measurements,46.0,NaN,21.5,50.0,606.0,NaN
910481,2025-12-19 23:59:04+00:00,smart_campus_room_measurements__lab_4_2__s00001,lab_4_2,smart_campus_room_measurements:row_00108940,smart_campus_room_measurements,46.0,NaN,21.1,50.0,570.0,NaN


In [54]:
# load comfort perceptions
comfort = pd.read_csv('../../data/raw/DATA_8/smart-campus-comfort-data/4_comfort_perception.csv')

# make sure timestamp is a normal column
if 'timestamp' not in df.columns and df.index.name == 'timestamp':
    df = df.reset_index()

# column names
df_ts_col = 'timestamp'
df_room_col = 'location_id'
comfort_ts_col = 'timestamp'
comfort_room_col = 'room'

# room mapping for the 4 known rooms
df_room_map = {
    'lab_2_2': 'lab_2_2',
    'lab_4_2': 'lab_4_2',
    'room_2_5': 'room_2_5',
    'room_2_12': 'room_2_12',
}
comfort_room_map = {
    'Lab 2.2': 'lab_2_2',
    'Lab 4.2': 'lab_4_2',
    'Room 2.5': 'room_2_5',
    'Room 2.12': 'room_2_12',
}

# parse timestamps
df = df.copy()
df[df_ts_col] = pd.to_datetime(df[df_ts_col], utc=True, errors='coerce')
comfort[comfort_ts_col] = pd.to_datetime(comfort[comfort_ts_col], utc=True, errors='coerce')

# build room key
df['room_key'] = df[df_room_col].map(df_room_map)
comfort['room_key'] = comfort[comfort_room_col].map(comfort_room_map)

# clean and sort
measurements = df.dropna(subset=[df_ts_col, 'room_key']).sort_values(df_ts_col)
comfort_clean = comfort.dropna(subset=[comfort_ts_col, 'room_key']).sort_values(comfort_ts_col)

# merge closest measurement for each rating within 10 minutes
max_gap = pd.Timedelta('4min')
df_with_comfort = pd.merge_asof(
    comfort_clean,
    measurements,
    left_on=comfort_ts_col,
    right_on=df_ts_col,
    by='room_key',
    direction='nearest',
    tolerance=max_gap,
    suffixes=('', '_meas'),
)

# keep only rows that got a measurement
df_with_comfort = df_with_comfort[df_with_comfort[df_room_col].notna()].copy()

# count and export
remaining_rows = len(df_with_comfort)
df_with_comfort.to_csv('../../data/processed/instant_mock.csv', index=False)
remaining_rows

1812